In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier,RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score,roc_curve,average_precision_score,brier_score_loss,
    accuracy_score,balanced_accuracy_score,precision_score,recall_score,
    confusion_matrix,classification_report,)

warnings.filterwarnings("ignore",category=FutureWarning)
pd.set_option("display.width",160)
pd.set_option("display.max_colwidth",90)
RANDOM_STATE=42
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore",sparse=False)

def build_preprocessor(cat_cols,num_cols):
    steps=[]
    if cat_cols:
        steps.append(("cat",make_ohe(),list(cat_cols)))
    if num_cols:
        steps.append(("num",StandardScaler(),list(num_cols)))
    return ColumnTransformer(steps)
print("Imports fine.")

In [ ]:
DATA_PATH=r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\individual_level_london.csv"
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape)
NEEDED=["survey_year","inactive","age_band","imd_decile","disab3","nssec5","gend3"]
OPTIONAL=["s_bin","borough","LAD24NM","readiness_opportunity","readiness_ability","wt_final","covid_affected","Eth7"]

missing=[c for c in NEEDED if c not in df.columns]
if missing:
    raise KeyError(f"Required columns missing from the file: {missing}")

present_optional=[c for c in OPTIONAL if c in df.columns]
absent_optional=[c for c in OPTIONAL if c not in df.columns]
print("required columns all present.")
print("optional columns present",present_optional)
print("optional columns absent ",absent_optional)

HAS_SBIN="s_bin" in df.columns
HAS_WEIGHTS="wt_final" in df.columns
HAS_READINESS={"readiness_opportunity","readiness_ability"}.issubset(df.columns)

print("rows per survey year:")
print(df["survey_year"].value_counts().sort_index())
print("issing values in the columns")
print(df[NEEDED+present_optional].isna().sum())

In [ ]:
'''checks if missingness is spread or present in one place'''

if "borough" in df.columns:
    complete=df.dropna(subset=NEEDED)
    by_borough=pd.DataFrame({"n_total": df.groupby("borough").size(),
        "n_complete": complete.groupby("borough").size(),}).fillna(0)
    by_borough["n_complete"]=by_borough["n_complete"].astype(int)
    by_borough["pct_retained"]=by_borough["n_complete"] / by_borough["n_total"]
    by_borough=by_borough.sort_values("pct_retained")

    overall_retention=len(complete) / len(df)
    print(f"removng missing demographics{overall_retention:.1%}")
    print("boroughs most affected by the removal")
    print(by_borough.head(8).round(4).to_string())
    spread=by_borough["pct_retained"].max()-by_borough["pct_retained"].min()
    print(f"how it is spread {spread:.1%}")
    if spread > 0.15:
        print("missingness is not evenly distributed")
       
    else:
        print("missingness is not affecting any results")
else:
    print("no borough")

In [ ]:
TRAIN_YEARS=["2016-17","2017-18","2018-19","2019-20","2020-21","2021-22"]
TEST_YEARS=["2022-23"]
EXCLUDE_BOROUGHS=["City of London"]

model_df=df.copy()
model_df["age_band_collapsed"]=(model_df["age_band"].astype(str).replace({"85+": "75+","75-84": "75+","nan": np.nan}))

DEMO_COLS=["age_band_collapsed","imd_decile","disab3","nssec5","gend3"]
CAT_COLS=["age_band_collapsed","disab3","nssec5","gend3"]
NUM_COLS=["imd_decile"]

m1=model_df.dropna(subset=["inactive"]+DEMO_COLS).copy()
m1["inactive"]=m1["inactive"].astype(int)

train_df=m1[m1["survey_year"].isin(TRAIN_YEARS)].copy()
test_df=m1[m1["survey_year"].isin(TEST_YEARS)].copy()

print(f"Complete {len(m1):,} of {len(df):,} rows ({len(m1)/len(df):.1%})")
print(f"Train (2016-17 to 2021-22){len(train_df):,}")
print(f"Test  (2022-23 {len(test_df):,}")
print(f"inactive rate, train: {train_df['inactive'].mean():.4f}")
print(f"inactive rate, test:  {test_df['inactive'].mean():.4f}")

# checking if removal skewed results or not
print("Inactive rate before dropping incomplete cases:", f"{df['inactive'].dropna().astype(int).mean():.4f}")
print("Inactive rate after :", f"{m1['inactive'].mean():.4f}")

In [ ]:
if HAS_READINESS:
    chk=model_df.copy()
    chk["has_readiness"]=chk["readiness_opportunity"].notna()
    print(f"readiness is there for {chk['has_readiness'].mean():.1%} of rows " f"({chk['has_readiness'].sum():,} of {len(chk):,})")
    print("inactivity rate by whether readiness was asked")
    print(chk.groupby("has_readiness")["inactive"].agg(["mean","size"]).round(4))
    print("readiness coverage by survey year ")
    print(chk.groupby("survey_year")["has_readiness"].mean().round(4))
    for c in ["age_band_collapsed","disab3","nssec5","gend3"]:
        comp=pd.crosstab(chk[c],chk["has_readiness"],normalize="columns").round(4)
        comp.columns=["no_readiness","has_readiness"]
        comp["diff"]=(comp["has_readiness"]-comp["no_readiness"]).round(4)
        biggest=comp["diff"].abs().max()
        print(f"{c}: biggest difference {biggest:.4f}")    
    print("more value means it is present more in a certain place")
else:
    print("No readiness columns")

In [ ]:
y_test=test_df["inactive"].values
prevalence=y_test.mean()
maj_acc=max(prevalence,1-prevalence)
print(f"most accuracy {maj_acc:.4f}   (AUC is 0.5000 by default")
print(f"inactivity present {prevalence:.4f}   ")
age_pipe=Pipeline([("prep",build_preprocessor(["age_band_collapsed"],[])),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),])
age_pipe.fit(train_df[["age_band_collapsed"]],train_df["inactive"])
p_age=age_pipe.predict_proba(test_df[["age_band_collapsed"]])[:,1]

BASELINE_AUC=roc_auc_score(y_test,p_age)
BASELINE_AP=average_precision_score(y_test,p_age)
print(f"logistic regression using age {BASELINE_AUC:.4f}, " f"average precision {BASELINE_AP:.4f}")

In [ ]:
def pick_threshold(y,p):
    fpr,tpr,thr=roc_curve(y,p)
    return float(thr[np.argmax(tpr-fpr)])
def evaluate(pipe,label,train_d,test_d,feature_cols,target="inactive",
 weight_col=None,store=None,verbose=True):
    X_tr,y_tr=train_d[feature_cols],train_d[target].astype(int).values
    X_te,y_te=test_d[feature_cols],test_d[target].astype(int).values

    fit_kwargs={}
    if weight_col and weight_col in train_d.columns:
        fit_kwargs["clf__sample_weight"]=train_d[weight_col].values

    pipe.fit(X_tr,y_tr,**fit_kwargs)
    p_tr=pipe.predict_proba(X_tr)[:,1]
    p_te=pipe.predict_proba(X_te)[:,1]
    thr=pick_threshold(y_tr,p_tr)
    pred_te=(p_te >= thr).astype(int)
    row={"label": label,
        "n_features": len(feature_cols),"train_auc": roc_auc_score(y_tr,p_tr),"test_auc": roc_auc_score(y_te,p_te),"auc_gap": roc_auc_score(y_tr,p_tr)-roc_auc_score(y_te,p_te),"test_ap": average_precision_score(y_te,p_te),"ap_floor": y_te.mean(),
        "brier": brier_score_loss(y_te,p_te),"thr": thr,"bal_acc": balanced_accuracy_score(y_te,pred_te),
        "recall": recall_score(y_te,pred_te,zero_division=0),"precision": precision_score(y_te,pred_te,zero_division=0),}

    if verbose:
        print(f"{label}")
        print(f"features ({len(feature_cols)}): {feature_cols}")
        print(f"train n={len(X_tr):,}   test n={len(X_te):,}")
        print(f"auc  train {row['train_auc']:.4f} | test {row['test_auc']:.4f} " f"| gap {row['auc_gap']:+.4f}")
        print(f"AP    test  {row['test_ap']:.4f} (floor {row['ap_floor']:.4f})")
        print(f"brier test  {row['brier']:.4f}")
        print(f"At threshold {thr:.3f}: balanced acc {row['bal_acc']:.4f}, "  f"recall {row['recall']:.4f}, precision {row['precision']:.4f}")
 
        print(confusion_matrix(y_te,pred_te))
    if store is not None:
        store.append(row)
    return pipe,p_te,row
results=[]


In [ ]:
# logistic regression using demohgrahics

lr_pipe=Pipeline([ ("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,C=1.0,random_state=RANDOM_STATE)),])
fitted_lr,p_lr,_=evaluate(lr_pipe,"A: Logistic regression, demographics",train_df,test_df,DEMO_COLS,store=results,)

In [ ]:
# imd using 10 categories

train_b=train_df.copy()
test_b=test_df.copy()
for d in (train_b,test_b):
    d["imd_cat"]=d["imd_decile"].astype(int).astype(str)

def add_interactions(tr,te,base_col,cat_col,prefix):
    cats=sorted(tr[cat_col].dropna().unique())
    made=[]
    for cat in cats:
        col=f"{prefix}_{cat}"
        tr[col]=tr[base_col] * (tr[cat_col] == cat).astype(int)
        te[col]=te[base_col] * (te[cat_col] == cat).astype(int)
        made.append(col)
    return made

int_cols=[]
int_cols += add_interactions(train_b,test_b,"imd_decile","disab3","imd_x_disab")
int_cols += add_interactions(train_b,test_b,"imd_decile","age_band_collapsed","imd_x_age")
print(f"Added {len(int_cols)} interaction columns.")

cat_b=["age_band_collapsed","disab3","nssec5","gend3","imd_cat"]
num_b=["imd_decile"]+int_cols
feat_b=cat_b+num_b
lr_rich=Pipeline([("prep",build_preprocessor(cat_b,num_b)),("clf",LogisticRegression(max_iter=3000,C=1.0,random_state=RANDOM_STATE)),])
fitted_lr_rich,p_lr_rich,_=evaluate(lr_rich,"B: Logistic regression, IMD as categorical + interactions",train_b,test_b,feat_b,store=results,)

In [ ]:
# gradient boost

gb_loose=Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=500,learning_rate=0.1,max_leaf_nodes=63,min_samples_leaf=5,l2_regularization=0.0,early_stopping=False,random_state=RANDOM_STATE)),])
fitted_gb_loose,p_gb_loose,_=evaluate(gb_loose,"no constraints for gradient boosting",train_df,test_df,DEMO_COLS,store=results,)

gb_tight=Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),])
fitted_gb,p_gb,_=evaluate(gb_tight,"gradient boosting stop early and regularised",train_df,test_df,DEMO_COLS,store=results,)
n_used=fitted_gb.named_steps["clf"].n_iter_
print(f"stopped {n_used} ")

In [ ]:
# model with baseline

summary=pd.DataFrame(results)
summary.insert(1,"beats_age_only",(summary["test_auc"]-BASELINE_AUC).round(4))
show=summary[["label","train_auc","test_auc","auc_gap","beats_age_only","test_ap","ap_floor","brier","bal_acc"]]
print(f"Reference: age-only baseline test AUC = {BASELINE_AUC:.4f}, " f"majority-class AUC = 0.5000")
print(show.round(4).to_string(index=False))
print("large positive gap means overfiited model")

In [ ]:
def make_gb():
    return Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),])
def make_lr():
    return Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),])
def rolling_origin(make_pipe,d,feature_cols,target="inactive",min_train_years=2):
    years=sorted(d["survey_year"].dropna().unique())
    rows=[]
    for i in range(min_train_years,len(years)):
        tr=d[d["survey_year"].isin(years[:i])]
        te=d[d["survey_year"] == years[i]]
        if len(te) < 100 or te[target].nunique() < 2 or tr[target].nunique() < 2:
            continue
        pipe=make_pipe()
        pipe.fit(tr[feature_cols],tr[target].astype(int))
        p=pipe.predict_proba(te[feature_cols])[:,1]
        y=te[target].astype(int).values
        rows.append({"trained_through": years[i-1],
            "tested_on": years[i],"n_train": len(tr),
            "n_test": len(te),"test_auc": roc_auc_score(y,p),
            "test_ap": average_precision_score(y,p),"prevalence": y.mean(),})
    return pd.DataFrame(rows)

roll_gb=rolling_origin(make_gb,m1,DEMO_COLS)
roll_lr=rolling_origin(make_lr,m1,DEMO_COLS)

print(" gradient boosting")
print(roll_gb.round(4).to_string(index=False))
print("logistic regression")
print(roll_lr.round(4).to_string(index=False))
print(f"GB  test AUC across folds: mean {roll_gb['test_auc'].mean():.4f}, " f"sd {roll_gb['test_auc'].std():.4f}, " f"range {roll_gb['test_auc'].min():.4f} to {roll_gb['test_auc'].max():.4f}")
print(f"LR  test AUC across folds: mean {roll_lr['test_auc'].mean():.4f}, "f"sd {roll_lr['test_auc'].std():.4f}, "f"range {roll_lr['test_auc'].min():.4f} to {roll_lr['test_auc'].max():.4f}")

In [ ]:
# confidence intervals are bootsrapped
def bootstrap_auc_ci(y,p,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed)
    y,p=np.asarray(y),np.asarray(p)
    out=[]
    for _ in range(n_boot):
        idx=rng.integers(0,len(y),len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        out.append(roc_auc_score(y[idx],p[idx]))
    out=np.array(out)
    return out.mean(),np.percentile(out,2.5),np.percentile(out,97.5)

def bootstrap_auc_delta(y,p_a,p_b,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed)
    y,p_a,p_b=np.asarray(y),np.asarray(p_a),np.asarray(p_b)
    out=[]
    for _ in range(n_boot):
        idx=rng.integers(0,len(y),len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        out.append(roc_auc_score(y[idx],p_b[idx])-roc_auc_score(y[idx],p_a[idx]))
    out=np.array(out)
    return out.mean(),np.percentile(out,2.5),np.percentile(out,97.5)

for name,p in [("  logistic regression",p_lr),(" lr + interactions",p_lr_rich),(" gradient boosting",p_gb)]:
    m,lo,hi=bootstrap_auc_ci(y_test,p)
    print(f"{name:24s} test AUC {m:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
m,lo,hi=bootstrap_auc_delta(y_test,p_lr,p_gb)
print(f"gb minus lr {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")


In [ ]:
fractions=[0.05,0.1,0.2,0.4,0.6,0.8,1.0]
curve=[]
for frac in fractions:
    sub=train_df.sample(frac=frac,random_state=RANDOM_STATE)
    if sub["inactive"].nunique() < 2:
        continue
    pipe=make_gb()
    pipe.fit(sub[DEMO_COLS],sub["inactive"])
    p_sub_tr=pipe.predict_proba(sub[DEMO_COLS])[:,1]
    p_sub_te=pipe.predict_proba(test_df[DEMO_COLS])[:,1]
    curve.append({"frac": frac,
        "n_train": len(sub),"train_auc": roc_auc_score(sub["inactive"],p_sub_tr),"test_auc": roc_auc_score(y_test,p_sub_te),})

curve_df=pd.DataFrame(curve)
curve_df["gap"]=curve_df["train_auc"]-curve_df["test_auc"]
print(curve_df.round(4).to_string(index=False))
last_three=curve_df["test_auc"].tail(3).values
print(f" test AUC {last_three[-1] - last_three[0]:+.4f}")


In [ ]:
# calibaration
frac_pos,mean_pred=calibration_curve(y_test,p_gb,n_bins=10,strategy="quantile")
cal=pd.DataFrame({"mean_predicted_risk": mean_pred,"actual_inactive_rate": frac_pos,"difference": frac_pos-mean_pred,})
print("calibration, gradient boosting model")
print(cal.round(4).to_string(index=False))
print(f"brier score: {brier_score_loss(y_test, p_gb):.4f}")
print(f"{brier_score_loss(y_test, np.full_like(p_gb, prevalence)):.4f}")
print(f"max cal error {cal['difference'].abs().max():.4f}")
#refit model
lr_bal=Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=RANDOM_STATE)),])
lr_bal.fit(train_df[DEMO_COLS],train_df["inactive"])
p_bal=lr_bal.predict_proba(test_df[DEMO_COLS])[:,1]
print(f"  AUC   {roc_auc_score(y_test, p_bal):.4f}  (basically unchanged)")
print(f"  brier {brier_score_loss(y_test, p_bal):.4f}  (much worse)")
print(f"  mean predicted risk {p_bal.mean():.4f} vs actual rate {prevalence:.4f}")


In [ ]:
# permutation

perm=permutation_importance(fitted_gb,test_df[DEMO_COLS],test_df["inactive"],scoring="roc_auc",n_repeats=10,random_state=RANDOM_STATE,n_jobs=1,)
imp=(pd.DataFrame({"feature": DEMO_COLS,"auc_drop": perm.importances_mean,"sd": perm.importances_std,}).sort_values("auc_drop",ascending=False).reset_index(drop=True))
print("permutation importance ")
print(imp.round(4).to_string(index=False))

In [ ]:
# checking if s bin improves anything

if HAS_SBIN:
    m1s=m1.dropna(subset=["s_bin"]).copy()
    tr_s=m1s[m1s["survey_year"].isin(TRAIN_YEARS)].copy()
    te_s=m1s[m1s["survey_year"].isin(TEST_YEARS)].copy()
    y_s=te_s["inactive"].astype(int).values
    print(f"rows with s_bin {len(m1s):,} of {len(m1):,} "f"({len(m1s)/len(m1):.1%})")
    print(f"Train {len(tr_s):,} | Test {len(te_s):,}")
    sbin_results=[]
    _,p_nos,_=evaluate( make_gb(),"D1: GB, demographics only (s_bin-complete rows)",tr_s,te_s,DEMO_COLS,store=sbin_results,)
    _,p_yess,_=evaluate(Pipeline([("prep",build_preprocessor(CAT_COLS+["s_bin"],NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),]),"D2: GB, demographics + s_bin",tr_s,te_s,DEMO_COLS+["s_bin"],store=sbin_results,)

    d_mean,d_lo,d_hi=bootstrap_auc_delta(y_s,p_nos,p_yess)
    print(f"AUC change after s bin {d_mean:+.4f}  95% CI [{d_lo:+.4f}, {d_hi:+.4f}]")
    if d_lo <= 0 <= d_hi:
        print(" sbin does not improve individual level")
    else:
        print(" s_bin does shift prediction.")
    
    results.extend(sbin_results)
else:
    print("ignore")

In [ ]:
# adding borough and covid data to see if it makes chnage

place_cols,place_cats=list(DEMO_COLS),list(CAT_COLS)
if "borough" in m1.columns:
    place_cols,place_cats=place_cols+["borough"],place_cats+["borough"]
if "covid_affected" in m1.columns:
    m1["covid_flag"]=m1["covid_affected"].astype(str)
    train_df["covid_flag"]=train_df["covid_affected"].astype(str)
    test_df["covid_flag"]=test_df["covid_affected"].astype(str)
    place_cols,place_cats=place_cols+["covid_flag"],place_cats+["covid_flag"]

if len(place_cols) > len(DEMO_COLS):
    print(f"extra columns  {[c for c in place_cols if c not in DEMO_COLS]}")
    if "borough" in m1.columns:
        print(f"Distinct boroughs: {m1['borough'].nunique()}")

    place_store=[]
    _,p_base,_=evaluate(make_gb(),"G1: GB, demographics only (reference)",train_df,test_df,DEMO_COLS,store=place_store,verbose=False,)
    _,p_place,_=evaluate(Pipeline([("prep",build_preprocessor(place_cats,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),]),"G2: GB, demographics + borough/covid",train_df,test_df,place_cols,store=place_store,verbose=False,)
    print(pd.DataFrame(place_store)[["label","n_features","train_auc","test_auc","auc_gap"]].round(4).to_string(index=False))
    
    d_mean,d_lo,d_hi=bootstrap_auc_delta(y_test,p_base,p_place)
    print(f"AUC change  {d_mean:+.4f}  95% CI [{d_lo:+.4f}, {d_hi:+.4f}]")
    if d_lo <= 0 <= d_hi:
        print("doesn't add meaningful information")
    elif d_hi < 0:
        print("negatively significant")
    else:
        print("Significantly positive")
    results.extend(place_store)
else:
    print("ignore")

In [ ]:
'''which decile are more at risk'''
risk=pd.DataFrame({"p": p_gb,"inactive": y_test})
risk["decile"]=pd.qcut(risk["p"].rank(method="first"),10,labels=range(1,11)).astype(int)

lift=(risk.groupby("decile").agg(n=("inactive","size"),mean_predicted=("p","mean"),actual_rate=("inactive","mean"),n_inactive=("inactive","sum")).sort_index(ascending=False))
lift["lift_vs_base"]=lift["actual_rate"] / prevalence
lift["cum_pct_population"]=(lift["n"].cumsum() / lift["n"].sum())
lift["cum_pct_inactive_captured"]=(lift["n_inactive"].cumsum() / lift["n_inactive"].sum())

print(f"inactivity rate in test year {prevalence:.4f}")

print(lift.round(4).to_string())
top2=lift["cum_pct_inactive_captured"].iloc[1]
top3=lift["cum_pct_inactive_captured"].iloc[2]
print(f"top 20% {top2:.1%} ")
print(f"top 30% {top3:.1%}.")


In [ ]:
#old code that had leakage

SOLO_CERTAIN,SOLO_SUSPECT,GROUP_FLAG=0.95,0.80,0.98
def leakage_screen(d,feature_cols,target):
    y=d[target].astype(int).values
    rows=[]
    for c in feature_cols:
        s=d[c]
        if pd.api.types.is_numeric_dtype(s):
            score=s.fillna(s.median()).astype(float).values
        else:
            rate=d.groupby(c)[target].mean()
            score=s.map(rate).fillna(y.mean()).astype(float).values
        if len(np.unique(y)) < 2 or np.all(score == score[0]):
            continue
        a=max(roc_auc_score(y,score),1-roc_auc_score(y,score))
        verdict=("leakage" if a >= SOLO_CERTAIN
                   else "check" if a >= SOLO_SUSPECT else "ok")
        rows.append({"feature": c,"solo_auc": a,"verdict": verdict})
    return (pd.DataFrame(rows).sort_values("solo_auc",ascending=False).reset_index(drop=True))
def group_leakage_check(tr,te,feature_cols,target,cat_cols,num_cols):
    pipe=Pipeline([("prep",build_preprocessor(cat_cols,num_cols)),("clf",HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=31,random_state=RANDOM_STATE)),])
    pipe.fit(tr[feature_cols],tr[target].astype(int))
    auc=roc_auc_score(te[target].astype(int),pipe.predict_proba(te[feature_cols])[:,1])
    verdict=("leakage" if auc >= GROUP_FLAG
               else "check" if auc >= 0.90 else "ok")
    print(f"check {auc:.4f}   {verdict}")
    return auc,verdict

print(leakage_screen(train_df,DEMO_COLS,"inactive").round(4).to_string(index=False))
group_leakage_check(train_df,test_df,DEMO_COLS,"inactive",CAT_COLS,NUM_COLS)

In [ ]:
if HAS_READINESS:
    bar=model_df.dropna(subset=["readiness_opportunity","readiness_ability"]+DEMO_COLS).copy()
    bar_tr=bar[bar["survey_year"].isin(TRAIN_YEARS)].copy()
    bar_te=bar[bar["survey_year"].isin(TEST_YEARS)].copy()
    avg_opp=bar_tr["readiness_opportunity"].mean()
    avg_abi=bar_tr["readiness_ability"].mean()

    for d in (bar_tr,bar_te):
        d["dominant_barrier"]=np.where((avg_opp-d["readiness_opportunity"]) > (avg_abi-d["readiness_ability"]),"opportunity","ability")
        d["target"]=(d["dominant_barrier"]=="opportunity").astype(int)

    leaky_feats=DEMO_COLS+["readiness_opportunity","readiness_ability"]
    leaky_num=NUM_COLS+["readiness_opportunity","readiness_ability"]

    print(leakage_screen(bar_tr,leaky_feats,"target").round(4).to_string(index=False))
    group_leakage_check(bar_tr,bar_te,leaky_feats,"target",CAT_COLS,leaky_num)

    leak_store=[]
    evaluate(Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS+["readiness_opportunity","readiness_ability"])),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),]),bar_tr,bar_te,leaky_feats,target="target",store=leak_store,)
   
else:
    print("ignore")

In [ ]:
# trying to improve the keakage model

if HAS_READINESS:
    opp=model_df.dropna(subset=["readiness_opportunity"]+DEMO_COLS).copy()
    opp_tr_raw=opp[opp["survey_year"].isin(TRAIN_YEARS)]
    dist=opp_tr_raw["readiness_opportunity"].value_counts(normalize=True).sort_index()
    print(dist.round(4).to_string())

    candidates=sorted(opp_tr_raw["readiness_opportunity"].dropna().unique())[:-1]
    options=[(c,(opp_tr_raw["readiness_opportunity"] <= c).mean()) for c in candidates]
    for c,rate in options:
        print(f"  <= {c:g}  ->  {rate:.4f}")
    CUT=min(options,key=lambda t: abs(t[1]-1/3))[0]
    chosen_rate=dict(options)[CUT]
    
    print(f"readiness_opportunity <= {CUT:g} uses {chosen_rate:.4f}")
    if not 0.15 <= chosen_rate <= 0.55:
        print("just trying")
      
    opp["low_opportunity"]=(opp["readiness_opportunity"] <= CUT).astype(int)
    opp_tr=opp[opp["survey_year"].isin(TRAIN_YEARS)].copy()
    opp_te=opp[opp["survey_year"].isin(TEST_YEARS)].copy()

    print(f"train {len(opp_tr):,} | Test {len(opp_te):,}")
    print(f"positive rate, train {opp_tr['low_opportunity'].mean():.4f} | "
f"test {opp_te['low_opportunity'].mean():.4f}")

    feats_opp=DEMO_COLS+(["s_bin"] if HAS_SBIN else [])
    cats_opp=CAT_COLS+(["s_bin"] if HAS_SBIN else [])
    opp_tr=opp_tr.dropna(subset=feats_opp)
    opp_te=opp_te.dropna(subset=feats_opp)

    print(leakage_screen(opp_tr,feats_opp,"low_opportunity").round(4).to_string(index=False))
    group_leakage_check(opp_tr,opp_te,feats_opp,"low_opportunity",cats_opp,NUM_COLS)

    opp_store=[]
    evaluate(Pipeline([("prep",build_preprocessor(cats_opp,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),]),"E1: LR, low opportunity readiness from demographics",opp_tr,opp_te,feats_opp,target="low_opportunity",store=opp_store,)
    _,p_opp,_=evaluate(Pipeline([("prep",build_preprocessor(cats_opp,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),]),"E2: GB, low opportunity readiness from demographics",opp_tr,opp_te,feats_opp,target="low_opportunity",store=opp_store,)

    y_opp=opp_te["low_opportunity"].values
    m,lo,hi=bootstrap_auc_ci(y_opp,p_opp)
    print(f" auc {m:.4f}, 95% ci [{lo:.4f}, {hi:.4f}]")
    results.extend(opp_store)
else:
    print("ignore")

In [ ]:
#sensitivity check
if HAS_WEIGHTS:
    tr_w=train_df.dropna(subset=["wt_final"])
    te_w=test_df.dropna(subset=["wt_final"])
    w_store=[]
    evaluate(make_lr(),"F1: LR, unweighted (reference)",tr_w,te_w,DEMO_COLS,store=w_store,verbose=False)
    evaluate(make_lr(),"F2: LR, survey-weighted fit",tr_w,te_w,DEMO_COLS,weight_col="wt_final",store=w_store,verbose=False)
    wdf=pd.DataFrame(w_store)[["label","train_auc","test_auc","brier"]]
    print(wdf.round(4).to_string(index=False))
    results.extend(w_store)
else:
    print("ignore")

In [ ]:
#saving table

final=pd.DataFrame(results)
final["beats_age_only"]=(final["test_auc"]-BASELINE_AUC).round(4)
cols=["label","n_features","train_auc","test_auc","auc_gap","beats_age_only","test_ap","ap_floor","brier","bal_acc","recall","precision"]
final=final[cols].round(4)

print(f"using age only {BASELINE_AUC:.4f} ")
print(final.to_string(index=False))

OUT=r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\model_results_v2.csv"
final.to_csv(OUT,index=False)
print(f" {OUT}")

In [ ]:
# checks individual risk minus inactivity

if "borough" in test_df.columns:
    bor=test_df[["borough","inactive"]].copy()
    bor["expected"]=p_gb
    if HAS_WEIGHTS:
        bor["w"]=test_df["wt_final"].values
    else:
        bor["w"]=1.0

    n_before=bor["borough"].nunique()
    bor=bor[~bor["borough"].isin(EXCLUDE_BOROUGHS)].copy()
    print(f" {EXCLUDE_BOROUGHS}: {n_before} boroughs  {bor['borough'].nunique()}")

    def wmean(s,w):
        return np.average(s,weights=w) if w.sum() > 0 else np.nan

    rows=[]
    for b,g in bor.groupby("borough"):
        obs=wmean(g["inactive"].values,g["w"].values)
        exp=wmean(g["expected"].values,g["w"].values)
        
        n_eff=(g["w"].sum()**2)/(g["w"]**2).sum()  
        se=np.sqrt(max(obs*(1-obs),1e-9)/max(n_eff,1))
        rows.append({"borough": b,"n":len(g),"n_effective": n_eff,"observed": obs,"expected": exp,"residual": obs-exp,"se": se})
    bres=pd.DataFrame(rows)
#bayes shrinkage
    tau2=max(0.0,bres["residual"].var(ddof=1)-(bres["se"] ** 2).mean())
    bres["shrinkage_weight"]=tau2 / (tau2+bres["se"] ** 2)
    bres["residual_shrunk"]=bres["residual"] * bres["shrinkage_weight"]
    bres["ci_lo"]=bres["residual"]-1.96 * bres["se"]
    bres["ci_hi"]=bres["residual"]+1.96 * bres["se"]
    bres["excess_significant"]=(bres["ci_lo"] > 0) | (bres["ci_hi"] < 0)
    bres=bres.sort_values("residual_shrunk",ascending=False).reset_index(drop=True)
    print(f" {tau2:.6f}")
    print(f"sd {np.sqrt(tau2):.4f}")
    print(f"shrinkage {bres['shrinkage_weight'].mean():.3f} ")
    print(f"shrinkage differs" f"{int(bres['excess_significant'].sum())} of {len(bres)}")
    print(bres[["borough","n","observed","expected","residual","residual_shrunk","excess_significant"]].round(4).to_string(index=False))

    if "borough" in df.columns:
        low_retention=set(by_borough.sort_values("pct_retained").head(8).index)
        flagged=set(bres.loc[bres["excess_significant"],"borough"])
        overlap=low_retention & flagged
        if overlap:
            print(f" {sorted(overlap)}")
else:
    print("ignore")

In [ ]:

if "borough" in test_df.columns:
    rank_corr=bres["observed"].corr(bres["residual_shrunk"],method="spearman")
    print(f"spearman correlation {rank_corr:.4f}")
  
    if abs(rank_corr) < 0.7:
        print(" different ranking")
    else:
        print("Rankings are similar")
       
    biggest_movers=bres.copy()
    biggest_movers["rank_observed"]=biggest_movers["observed"].rank(ascending=False)
    biggest_movers["rank_residual"]=biggest_movers["residual_shrunk"].rank(ascending=False)
    biggest_movers["rank_change"]=(biggest_movers["rank_observed"]- biggest_movers["rank_residual"])
    movers=biggest_movers.reindex(biggest_movers["rank_change"].abs().sort_values(ascending=False).index).head(8)
    print(movers[["borough","observed","expected","rank_observed","rank_residual","rank_change"]].round(4).to_string(index=False))

    OUT_B=(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data" r"\borough_demand_residuals_v2.csv")
    bres.round(6).to_csv(OUT_B,index=False)
    print(f" {OUT_B}")

In [ ]:
def subgroup_audit(d,y_true,y_prob,threshold,group_cols):
    out=[]
    for gc in group_cols:
        if gc not in d.columns:
            continue
        for level,idx in d.groupby(gc,observed=True).groups.items():
            pos=d.index.get_indexer(idx)
            yy,pp=y_true[pos],y_prob[pos]
            if len(yy) < 100 or len(np.unique(yy))<2:
                continue
            pred=(pp >= threshold).astype(int)
            tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
            out.append({"attribute": gc,"group": str(level),"n": len(yy),
                "base_rate": yy.mean(),"auc": roc_auc_score(yy,pp),"mean_pred": pp.mean(),"calib_error": pp.mean()-yy.mean(),
                "fnr": fn / (fn+tp) if (fn+tp) else np.nan,"fpr": fp / (fp+tn) if (fp+tn) else np.nan,"selection_rate": pred.mean(),})
    return pd.DataFrame(out)

aud=test_df.reset_index(drop=True).copy()
aud["imd_tertile"]=pd.qcut(aud["imd_decile"],3,labels=["most deprived","middle","least deprived"])
THRESH=pick_threshold(train_df["inactive"].values,fitted_gb.predict_proba(train_df[DEMO_COLS])[:,1])

audit=subgroup_audit(aud,y_test,p_gb,THRESH,["disab3","gend3","age_band_collapsed","imd_tertile","nssec5"])
print(f"threshold {THRESH:.4f}")
print(audit.round(4).to_string(index=False))
for attr in audit["attribute"].unique():
    sub=audit[audit["attribute"] == attr]
    print(f"{attr:20s} auc spread {sub['auc'].max() - sub['auc'].min():.4f} | "f"fnr spread {sub['fnr'].max() - sub['fnr'].min():.4f} | " f"error {sub['calib_error'].abs().max():.4f}")


In [ ]:
#checking if threshold choice was good

target_fnr=audit["fnr"].median()
print(f"median fnr {target_fnr:.4f}")
rows=[]
for attr in ["disab3","gend3","imd_tertile"]:
    if attr not in aud.columns:
        continue
    for level,idx in aud.groupby(attr,observed=True).groups.items():
        pos=aud.index.get_indexer(idx)
        yy,pp=y_test[pos],p_gb[pos]
        if len(yy) < 100 or yy.sum() < 20:
            continue
        # threshold at which this group's fnr equals the target
        cand=np.quantile(pp[yy == 1],target_fnr)
        pred=(pp >= THRESH).astype(int)
        tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
        rows.append({"attribute": attr,"group": str(level),"n": len(yy),
                     "fnr_at_global_thresh": fn / (fn+tp),"thresh_for_equal_fnr": cand,"shift_needed": cand-THRESH})
eq=pd.DataFrame(rows)
print(eq.round(4).to_string(index=False))


In [ ]:
# checks auc within each group

overall_auc=roc_auc_score(y_test,p_gb)
print(f" test auc {overall_auc:.4f}")
rows=[]
for attr in ["nssec5","disab3","imd_tertile","age_band_collapsed"]:
    if attr not in aud.columns:
        continue
    sub=audit[audit["attribute"] == attr]
    if not len(sub):
        continue
    w=sub["n"] / sub["n"].sum()
    rows.append({
        "held_constant": attr,"n_groups": len(sub),
        "weighted_within_group_auc": float((sub["auc"] * w).sum()),"min_group_auc": sub["auc"].min(),"max_group_auc": sub["auc"].max(),})
wg=pd.DataFrame(rows)
wg["retained_vs_overall"]=(wg["weighted_within_group_auc"]-0.5) / (overall_auc-0.5)
print(wg.round(4).to_string(index=False))

worst=wg.loc[wg["retained_vs_overall"].idxmin()]
print(f" {worst['held_constant']}  "
      f"{worst['retained_vs_overall']:.0%} of the discrimination.")



In [ ]:


def net_benefit(y,p,pt):
    pred=(p>=pt).astype(int)
    tp=((pred==1)&(y == 1)).sum()
    fp=((pred==1)&(y==0)).sum()
    n=len(y)
    return tp/n-(fp/n)*(pt/(1-pt))

thresholds=np.arange(0.05,0.61,0.025)
imd_only=Pipeline([("prep",build_preprocessor([],["imd_decile"])),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))])
imd_only.fit(train_df[["imd_decile"]],train_df["inactive"])
p_imd=imd_only.predict_proba(test_df[["imd_decile"]])[:,1]

dca=[]
for pt in thresholds:
    dca.append({"threshold": pt,"nb_model": net_benefit(y_test,p_gb,pt),"nb_imd_only": net_benefit(y_test,p_imd,pt),
        "nb_treat_all": prevalence-(1-prevalence) * (pt / (1-pt)),"nb_treat_none": 0.0,})
dca=pd.DataFrame(dca)
dca["model_best"]=((dca["nb_model"] > dca["nb_treat_all"]) &(dca["nb_model"] > dca["nb_treat_none"]) &(dca["nb_model"] > dca["nb_imd_only"]))
print(dca.round(4).to_string(index=False))

useful=dca[dca["model_best"]]
if len(useful):
    print(f"{useful['threshold'].min():.3f} to {useful['threshold'].max():.3f}.")

else:
    print("does not dominate")


In [ ]:
#main table

order=np.argsort(-p_gb)
y_sorted=y_test[order]
total_inactive=y_sorted.sum()

cap_rows=[]
for pct in [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]:
    k=int(len(y_sorted) * pct)
    reached=y_sorted[:k]
    cap_rows.append({"reach_pct": pct,"people_reached": k,"inactive_reached": int(reached.sum()), "pct_of_inactive_captured": reached.sum() / total_inactive,"precision": reached.mean(),"lift_vs_random": (reached.sum() / total_inactive) / pct,})
cap=pd.DataFrame(cap_rows)
print(f"test year {len(y_sorted):,} adults, {int(total_inactive):,} inactive "f"({prevalence:.1%})")
print(cap.round(4).to_string(index=False))
best=cap.loc[cap["lift_vs_random"].idxmax()]
print(f"efficiency is highest  at {best['reach_pct']:.0%}  "f"{best['lift_vs_random']:.2f} ")

OUT_C=(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data"r"\targeting_capacity_v2.csv")
cap.round(6).to_csv(OUT_C,index=False)
print(f"\n {OUT_C}")

In [ ]:
#model card

card={"train_n": len(train_df),"test_n": len(test_df),"train_years": f"{TRAIN_YEARS[0]} to {TRAIN_YEARS[-1]}",
    "test_year": TEST_YEARS[0],"features": ", ".join(DEMO_COLS),"test_auc": roc_auc_score(y_test,p_gb),"test_ap": average_precision_score(y_test,p_gb),
    "ap_floor": prevalence,"brier": brier_score_loss(y_test,p_gb),"brier_base_rate": brier_score_loss(y_test,np.full_like(p_gb,prevalence)),"rolling_auc_mean": roll_gb["test_auc"].mean(),
    "rolling_auc_sd": roll_gb["test_auc"].std(),"age_only_baseline_auc": BASELINE_AUC,"deployment_threshold": THRESH,"capture_at_20pct_reach": cap.loc[cap["reach_pct"] == 0.20,"pct_of_inactive_captured"].iloc[0],
    "max_fnr_spread": audit.groupby("attribute")["fnr"].apply(lambda s: s.max()-s.min()).max(),"complete_case_rate": len(m1) / len(df),}
for k,v in card.items():
    print(f"{k:28s} {v:.4f}" if isinstance(v,float) else f"{k:28s} {v}")

OUT_D=(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data"r"\model_card_v2.csv")
pd.DataFrame([card]).to_csv(OUT_D,index=False)
print(f"\n {OUT_D}")